# Customer Segmentation — K-Means
Educational project with synthetic customer data. Run cells in order.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
DATA = Path("../data/customers.csv") if Path("../data/customers.csv").exists() else Path("data/customers.csv")
df = pd.read_csv(DATA)
df.head()

## 1. Explore the data

In [ ]:
print(df.shape)
print(df.isna().sum())
display(df.describe())
df[["RecencyDays","Frequency","Monetary"]].hist(figsize=(10,3))
plt.tight_layout(); plt.show()

## 2. Scale features before K-Means

In [ ]:
features = ["RecencyDays", "Frequency", "Monetary"]
X = StandardScaler().fit_transform(df[features])

## 3. Compare cluster counts

In [ ]:
ks = range(2,9)
inertias, silhouettes = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))
plt.plot(list(ks), inertias, marker="o"); plt.xlabel("K"); plt.ylabel("Inertia"); plt.title("Elbow Method"); plt.show()
plt.plot(list(ks), silhouettes, marker="o"); plt.xlabel("K"); plt.ylabel("Silhouette"); plt.show()

## 4. Fit and interpret clusters

In [ ]:
best_k = list(ks)[silhouettes.index(max(silhouettes))]
df["Cluster"] = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(X)
print("Best k:", best_k)
display(df.groupby("Cluster")[features].agg(["mean", "count"]).round(2))

## 5. PCA visualization (only for plotting)

In [ ]:
points = PCA(n_components=2).fit_transform(X)
plt.scatter(points[:,0],points[:,1],c=df["Cluster"],cmap="viridis",s=20)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Customer Segments"); plt.colorbar(label="Cluster"); plt.show()

## 6. Discussion
- Which segment has highest average spending?
- Does higher frequency correlate with lower recency?
- What would change with real data?
- Cluster labels are arbitrary; interpret the means rather than the label numbers.